# Week 4 Activity: Delay, Filters, Reverb

Complete this activity as part of your participation grade. Pending length of the lecture, you will have time in class to work. Everything you need to complete this activity can be found in this week's (or a previous week's) lecture code. For this activity, you may want to consult your Audio Tech I notes (or the Computer Music Tutorial)

In [6]:
import numpy as np
from scipy.io.wavfile import read
from scipy.signal import sawtooth, square, butter, filtfilt
import matplotlib.pyplot as plt
from IPython.display import Audio, Image

## Delay
1) Create a function that will modify a passed signal by adding a delay of $m$ milliseconds to the signal.

In [7]:



def delayfunc(m, signal='../audio/AcousticGuitar.wav'):
    fs, x = read(signal)
    # normalize between -1 and 1
    data = x/abs(x).max() # keep everything scaled and proportional
    copy = data.copy()
    offset = m # delay in ms
    offset_samples = int(fs/1000 * offset) # divide by 1000 then multiply by offset to convert ms to sec
    pad = np.zeros(offset_samples)
    orig = np.concatenate([data, pad]) # put a bunch of zeroes at end
    delay = np.concatenate([pad, copy]) # use copy to not confuse
    new_signal = orig + delay
    return new_signal

Audio(delayfunc(1000), rate=44100) #test run


2) Modify the function such that you can optionally scale the amplitude of the delayed signal 

In [12]:
def delayfunc(m, signal='../audio/AcousticGuitar.wav', amp=1):
    fs, x = read(signal)
    # normalize between -1 and 1
    data = x/abs(x).max() # keep everything scaled and proportional
    copy = data.copy()
    offset = m # delay in ms
    offset_samples = int(fs/1000 * offset) # divide by 1000 then multiply by offset to convert ms to sec
    pad = np.zeros(offset_samples)
    orig = np.concatenate([data, pad]) # put a bunch of zeroes at end
    delay = np.concatenate([pad, copy]) # use copy to not confuse
    new_delay = delay * amp
    new_signal = orig + new_delay
    return new_signal

Audio(delayfunc(1000, '../audio/AcousticGuitar.wav', 5), rate=44100) #test run


3) Modify the function so that the user can specify the number of delays to add to the original signal (and optionally, scale each subsequent delay amplitude by a factor of $1/n$

In [53]:
def delayfunc(m, signal='../audio/AcousticGuitar.wav', amp=1, repeats=1):
    fs, x = read(signal)
    repeats = int(repeats) # num times delay repeats
    # normalize between -1 and 1
    data = x/abs(x).max() # keep everything scaled and proportional
    copy = data.copy()
    offset = m # delay in ms
    #offset_samples = int(fs/1000 * offset) # divide by 1000 then multiply by offset to convert ms to sec
    #pad = np.zeros(offset_samples)
    delay = np.concatenate([copy, np.zeros(int(fs/1000*offset*repeats))])
    #delay = np.concatenate([data, pad])

    for i in range(repeats+1):
        pad_beg = np.zeros(int(fs/1000 * offset * i))
        pad_end = np.zeros(int(fs/1000 * offset * (repeats-i)))
        delay += np.concatenate([pad_beg, copy, pad_end])
        new_delay = delay * amp
        new_samp = delay + new_delay
    return new_samp

Audio(delayfunc(1000,'../audio/AcousticGuitar.wav',5,6), rate=44100) #test run

## Filters
1. Create a function that will apply an feedforward comb filter by computing the delay length based off a given resonant frequency in Hz.

2. Create a function that will apply an feedback comb filter by computing the delay length based off a given resonant frequency in Hz.

4. Use your comb filter functions on a wave from the audio folder. Try applying different resonant frequencies and delay lengths. How are the filter results different?

3. Create a function that will apply a butterworth filter to a signal with filter type options 'highpass', 'lowpass', 'bandpass', and 'bandstop'.

## Reverb/Convolution

1. Create a function that will apply a simple moving average filter by convolving the filter kernel and an incoming signal.

2. Apply your filter to a noise signal. What is the effect? What happens if you increase or decrease the kernel size?

3. Create a function that applies convolution reverb to an input signal given an impulse response (this can be default loaded from the audio files). Use np.convolve to create this function.

4. Create another function that applies convolution reverb to an input signal given an impulse response, but this time do not use np.convolve. You should write the convolution from scratch.

    Challenge yourself to create the most efficient function and time your implementation against np.convolve. 
    While developing and testing your function, do not use real audio files. Start with short signals (e.g., impulses, noise, or short sinusoids). Using long signals with loop-based implementations will result in extremely slow run times.

    **Hint:** in a similar manner to how you may have designed your delay function, recall the functions `numpy.zeros` and `numpy.roll` along with how to manipulate multidimensional numpy arrays (e.g., scalar product, summing columns with `vstack`, etc.) If you plan to try `numpy.roll` DO NOT use it in a loop for convolution with a real audio file! (You'll kill your memory), instead consider the `map` function. You may also want to check out the following function which is similar to numpy.roll but more efficient for this task: `scipy.linalg.circulant`.

    You may wish to visit [here](https://numpy.org/doc/stable/user/basics.broadcasting.html) for review of broadcasting (i.e., form some calculation across index value I and column value C) in numpy